# BB: BP-OSD decoding of unresolved shots

Decode stored shots for which RelayBP did not converge. Match the physical
conditions and primary RelayBP settings to the source run. Leave
`BPOSD_RETRY_SOURCE_HASH` empty to select the latest matching record.

The retry updates failure statistics without resampling. All matching
unresolved shots are processed; `SHOTS` does not limit the retry count.


## Parameters


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "code_construction" / "affine_codes.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

PYTHON = Path(sys.executable)
SCRIPT = REPO_ROOT / "bivariate_bicycle/simulate_memory.py"
SAMPLE_DIR = Path("bivariate_bicycle/results/samples")
DECODE_DIR = Path("bivariate_bicycle/results/decode")

# Physical conditions of the saved run.
CODES = ["bb_288_12_18"]
BASIS = "X"  # Z | X | both
DECODING_MODE = "xz"  # xyz | xz | both
P_LIST = [0.0045]
CYCLES = 8
SHOTS = 2_000_000
SHOT_CHUNK = 200

# Empty selects the latest matching source record.
BPOSD_RETRY_SOURCE_HASH = ""
FORCE_DECODE = False
PROGRESS_SECONDS = 30

# Primary RelayBP parameters must match the source record.
GAMMA0 = 0.1
PRE_ITER = 200
NUM_SETS = 20
SET_MAX_ITER = 100
GAMMA_DIST_MIN = -0.24
GAMMA_DIST_MAX = 0.6
STOP_NCONV = 1

# BPOSD retry parameters.
BPOSD_MAX_ITER = 300
BPOSD_BP_METHOD = "minimum_sum"
BPOSD_MS_SCALING_FACTOR = 0.0
BPOSD_SCHEDULE = "serial"
BPOSD_OSD_METHOD = "OSD_CS"
BPOSD_OSD_ORDER = 1
# BP-OSD parallelism uses worker processes; keep decoder threads at one.
BPOSD_THREADS = 1
BPOSD_WORKERS = max(1, min(8, os.cpu_count() or 1))

## Command


In [ ]:
def _csv(values):
    return ",".join(str(value) for value in values)


cmd = [
    str(PYTHON),
    str(SCRIPT),
    "--stage",
    "decode",
    "--decoder",
    "relaybp",
    "--codes",
    _csv(CODES),
    "--basis",
    BASIS,
    "--decoding-mode",
    DECODING_MODE,
    "--p-list",
    _csv(P_LIST),
    "--cycles",
    str(CYCLES),
    "--shots",
    str(SHOTS),
    "--shot-chunk",
    str(SHOT_CHUNK),
    "--sample-dir",
    str(SAMPLE_DIR),
    "--decode-dir",
    str(DECODE_DIR),
    "--progress-seconds",
    str(PROGRESS_SECONDS),
    "--gamma0",
    str(GAMMA0),
    "--pre-iter",
    str(PRE_ITER),
    "--num-sets",
    str(NUM_SETS),
    "--set-max-iter",
    str(SET_MAX_ITER),
    "--gamma-dist-min",
    str(GAMMA_DIST_MIN),
    "--gamma-dist-max",
    str(GAMMA_DIST_MAX),
    "--stop-nconv",
    str(STOP_NCONV),
    "--bposd-retry-unconverged",
    "--bposd-max-iter",
    str(BPOSD_MAX_ITER),
    "--bposd-bp-method",
    str(BPOSD_BP_METHOD),
    "--bposd-ms-scaling-factor",
    str(BPOSD_MS_SCALING_FACTOR),
    "--bposd-schedule",
    str(BPOSD_SCHEDULE),
    "--bposd-osd-method",
    str(BPOSD_OSD_METHOD),
    "--bposd-osd-order",
    str(BPOSD_OSD_ORDER),
    "--bposd-threads",
    str(BPOSD_THREADS),
    "--bposd-workers",
    str(BPOSD_WORKERS),
]
if BPOSD_RETRY_SOURCE_HASH:
    cmd += ["--bposd-retry-source-hash", str(BPOSD_RETRY_SOURCE_HASH)]
if FORCE_DECODE:
    cmd += ["--force-decode"]

print(" ".join(shlex.quote(part) for part in ["python", *cmd[1:]]))

## Run


In [ ]:
completed = subprocess.run(cmd, cwd=str(REPO_ROOT), text=True)
if completed.returncode:
    raise SystemExit(completed.returncode)

## Results


In [ ]:
summary_path = REPO_ROOT / DECODE_DIR / "summary.txt"
print(summary_path)
if summary_path.exists():
    print(summary_path.read_text()[-8000:])
else:
    print("summary.txt not found")